# Proposed validation — review before it runs

**What this measures:** The claim names MetaMathQA and this repo publishes comparable rows (76 result JSONs) from its own harness `method_comparison/MetaMathQA/run.py`, so the target is the raw harness field `num_trainable_params` from a real `ternary_adapt` run at the llama-3.2-3B-rank32 protocol, thresholded at 1,146,840 = published LoRA r=32 row (9,174,720) / 8 (derived TernaryAdapt ≈ 279,552, ~32.8× fewer), with `test_accuracy` from the same run guarding the "without losing fit" half. A NEW experiment config `experiments/ternary_adapt/llama-3.2-3B-rank32` is created for the new method, and `preimport: peft.tuners.ternary_adapt` is required because the PR's `register_peft_method` call and the dynamic `PeftType` member only execute on import of that package (peft/tuners/__init__.py does not import it), without which the harness cannot load `peft_type: TERNARY_ADAPT`.

**Target metric:** `num_trainable_params`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at ``, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

## What will run

This validation reuses **the repository's own benchmark** rather than a synthesized stand-in, so the numbers are comparable to the results this repo publishes.

- **runner**: `method_comparison/MetaMathQA/run.py`
- **experiments**: `experiments/ternary_adapt/llama-3.2-3B-rank32`
- **results read from**: `method_comparison/MetaMathQA/results/*.json`
- **comparable rows**: `ternary_adapt`
- **pre-imports before loading the config**: `peft.tuners.ternary_adapt`

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
# Suite kind (a): the repo's own MetaMathQA harness. The claim names MetaMathQA and this repo publishes comparable
# numbers from exactly this runner, so a synthesized CPU surrogate would answer a different question. The published
# LoRA r=32 corpus row IS the baseline; no baseline arm runs. Metric keys are copied character-for-character from
# the harness's result schema (num_trainable_params, test_accuracy).
benchmarks:
  - name: ternary-adapt-metamathqa-rank32
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/ternary_adapt/llama-3.2-3B-rank32  # NEW config dir for the NEW method (see EXPERIMENT_FILES)
        results_glob: "method_comparison/MetaMathQA/results/*.json"
        method: ternary_adapt
        # The PR confines registration to the new package's __init__ (register_peft_method + a dynamically added
        # PeftType member execute on import; peft/tuners/__init__.py does not import it), so the runner must
        # import peft.tuners.ternary_adapt in-process before loading adapter_config.json with peft_type TERNARY_ADAPT.
        preimport:
          - "peft.tuners.ternary_adapt"
      scorer: num_trainable_params
    metrics:
      # TARGET: "far fewer trainable parameters than standard LoRA" — raw harness field, lower is better.
      # Derivation: published LoRA r=32 q/v row = 28 x [32*(3072+3072) + 32*(3072+1024)] = 9,174,720 trainable.
      # TernaryAdapt default near-square Kronecker blocks = 28 x [(64*64+48*48) + (32*64+32*48)] = 279,552
      # (~32.8x fewer). Threshold 1,146,840 = 9,174,720 / 8: demands at least 8x fewer params than the published
      # LoRA row while sitting ~4.1x above the derived ternary value, so a correct implementation clears it.
      - name: num_trainable_params
        direction: min
        threshold: 1146840
        role: target
      # GUARDRAIL: "without losing fit on MetaMathQA" — the fit half of the claim, from the SAME harness run.
      # Floor 0.40 (test_accuracy is a 0-1 fraction in this harness's results JSONs): the published LoRA r=32 row
      # clears it comfortably (a successful MetaMathQA SFT of a 3B model), while a broken ternarization
      # (collapsed base weights or a dead mask) falls toward untuned-base 0-shot levels and fails, so this is a
      # real no-regression bound the baseline already satisfies.
      - name: test_accuracy
        direction: max
        threshold: 0.4
        role: guardrail
    baseline:
      source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
      values:
        num_trainable_params: 9174720
    policy:
      guardrail_veto: true
    compute:
      # ONE arm = llama-3.2-3B SFT at the harness's default MetaMathQA protocol (~2-5k optimizer steps x ~1.5 s/step
      # on one A100-class GPU) + generation-based test_accuracy eval + ~7 GB base-model download ≈ 2-4 h;
      # 21600 s adds honest download/scheduling headroom.
      tier: gpu
      timeout_s: 21600
    held_constant:
      - "base model meta-llama/Llama-3.2-3B at the llama-3.2-3B-rank32 protocol (same as every published corpus row)"
      - "training protocol: the harness's default training params (no per-experiment training_params.json override)"
      - "target modules q_proj+v_proj, identical to the comparable LoRA r=32 corpus row"
      - "MetaMathQA train/eval data and seed fixed by run.py, same for every published row"
    avoid:
      - "unpinned base-model revision"
      - "a per-experiment training_params.json that overrides the published default protocol and breaks comparability"
      - "pointing experiments at another method's directory (would measure that method, not ternary_adapt)"
    provenance:
      num_trainable_params: "user_guidance"
      test_accuracy: "user_guidance"
      baseline: "published_results:method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json (LoRA r=32 q/v row = 9,174,720 trainable, per the PR derivation)"
      suite: "repo_runner:method_comparison/MetaMathQA/run.py"
      held_constant: "protocol_doc:method_comparison/README.md"
      preimport: "upstream_comment:PR diff — registration runs only on import of peft.tuners.ternary_adapt"
      experiments: "synthesized: new config dir for the new method, mirroring experiments/adalora/llama-3.2-3B-rank32 shape"
```